# CSE 251B — Optimizer Ablation (AdamW vs Muon vs SOAP)

**Controlled experiment**: identical nanoGPT architecture and data, only the optimizer changes.

Set `OPTIMIZER` in cell 2, run all cells, log to W&B, repeat for each optimizer.

| `OPTIMIZER` | Description | Peak LR |
|---|---|---|
| `"adamw"` | Stock nanoGPT AdamW — the baseline | 6e-4 |
| `"muon_adam"` | Muon (2D block params) + AdamW (embed/1D) | 0.025 / 6e-4 |
| `"soap"` | SOAP (Shampoo as Adam Preconditioner) via heavyball | 3e-3 |

Everything else is **locked** across runs:
- Architecture: nanoGPT, n_layer=8, n_embd=768 (~96M params)
- Data: FineWebEDU 10 shards (~1B tokens)
- Steps: 5000, effective batch 131072 tokens/step, grad_clip=1.0
- LR schedule shape: cosine warmup→decay (same proportion, different peak per optimizer)

---
**Runtime**: T4 GPU or better


In [ ]:
# ── 1. GPU check + install ────────────────────────────────────────────────────
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or '⚠️ No GPU')

import torch
print(f'PyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  {p.total_memory/1e9:.1f} GB')

!pip install -q huggingface_hub wandb
# heavyball provides SOAP (Shampoo-based); only installed if OPTIMIZER="soap"
!pip install -q heavyball

In [ ]:
# ── 2. Config ─────────────────────────────────────────────────────────────────
# ↓↓↓ CHANGE THIS to switch optimizer ↓↓↓
OPTIMIZER = "muon_adam"  # @param ["adamw", "muon_adam", "soap"]
# ↑↑↑

from pathlib import Path
import math

USE_DRIVE = True  # @param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/cse251b')
else:
    BASE_DIR = Path('/content/cse251b')

DATA_DIR = BASE_DIR / 'data' / 'finewebedu10B'
CKPT_DIR = BASE_DIR / f'checkpoints_ablation_{OPTIMIZER}'
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = CKPT_DIR / f'{OPTIMIZER}_latest.pt'

# ── Shared training config (identical across all optimizer runs) ──────────────
MICRO_BATCH = 8
SEQ_LEN     = 1024
GRAD_ACCUM  = 16      # effective batch = 8 × 1024 × 16 = 131,072 tokens/step
NUM_STEPS   = 5000
GRAD_CLIP   = 1.0     # nanoGPT default
VAL_EVERY   = 250
CKPT_EVERY  = 500

# ── Per-optimizer hyperparams ─────────────────────────────────────────────────
# AdamW (baseline) — from nanoGPT config/train_gpt2.py
ADAMW_LR      = 6e-4
ADAMW_MIN_LR  = 6e-5    # 10% of peak, per nanoGPT
ADAMW_WD      = 1e-1
ADAMW_BETAS   = (0.9, 0.95)

# Muon — natural scale; much higher LR than AdamW because updates are normalized
MUON_LR  = 0.025
MUON_MIN_LR = 0.025 * 0.1
MUON_WD  = 0.0125
MUON_MU  = 0.95
# AdamW for non-Muon params (embed, lm_head, LayerNorm, biases) in muon_adam mode
MUON_ADAM_LR     = 6e-4
MUON_ADAM_MIN_LR = 6e-5

# SOAP — Shampoo as Adam Preconditioner (heavyball)
SOAP_LR     = 3e-3      # heavyball recommended default
SOAP_MIN_LR = 3e-4
SOAP_WD     = 1e-1
SOAP_BETAS  = (0.95, 0.95)
SOAP_PRECOND_FREQ = 10  # update preconditioner every N steps (↑ = faster, ↓ = better)

# LR warmup (same proportion across all optimizers)
WARMUP_ITERS = 200

# W&B
USE_WANDB      = False  # @param {type:"boolean"}
WANDB_PROJECT  = 'cse251b-group-gli'
WANDB_RUN_NAME = f'ablation-{OPTIMIZER}-v1'   # auto-named per optimizer

print(f'Optimizer : {OPTIMIZER}')
print(f'Ckpt dir  : {CKPT_DIR}')

In [ ]:
# ── 3. Download data (skip if cached) ────────────────────────────────────────
from huggingface_hub import hf_hub_download

NUM_TRAIN_SHARDS = 10
HF_REPO = 'kjj0/finewebedu10B-gpt2'

def maybe_download(fname):
    dest = DATA_DIR / fname
    if dest.exists(): print(f'  [cached] {fname}'); return
    print(f'  ↓ {fname}', end='', flush=True)
    hf_hub_download(repo_id=HF_REPO, filename=fname, repo_type='dataset', local_dir=str(DATA_DIR))
    print(' ✓')

maybe_download('finewebedu_val_000000.bin')
for i in range(1, NUM_TRAIN_SHARDS + 1):
    maybe_download(f'finewebedu_train_{i:06d}.bin')

train_files = sorted(DATA_DIR.glob('finewebedu_train_*.bin'))
val_files   = sorted(DATA_DIR.glob('finewebedu_val_*.bin'))
print(f'Train {len(train_files)} shards  Val {len(val_files)} shards')
assert train_files and val_files

In [ ]:
# ── 4. Data loader ────────────────────────────────────────────────────────────
import numpy as np

def load_shard(path):
    with open(path, 'rb') as f:
        hdr = np.frombuffer(f.read(256 * 4), dtype=np.int32)
        assert hdr[0] == 20240520 and hdr[1] == 1
        return np.frombuffer(f.read(int(hdr[2]) * 2), dtype=np.uint16).copy()

class DataLoader:
    def __init__(self, files, B, T):
        self.files, self.B, self.T = list(files), B, T
        self._fi, self._pos = 0, 0
        self._tok = load_shard(self.files[0])
    def __iter__(self): return self
    def __next__(self):
        B, T = self.B, self.T
        while self._pos + B * T + 1 > len(self._tok):
            self._fi = (self._fi + 1) % len(self.files)
            self._tok = load_shard(self.files[self._fi]); self._pos = 0
        buf = self._tok[self._pos : self._pos + B * T + 1]; self._pos += B * T
        x = torch.from_numpy(buf[:-1].astype(np.int64)).view(B, T).cuda()
        y = torch.from_numpy(buf[1:].astype(np.int64)).view(B, T).cuda()
        return x, y
    def state_dict(self): return {'fi': self._fi, 'pos': self._pos}
    def load_state_dict(self, s):
        self._fi = s['fi']; self._tok = load_shard(self.files[self._fi]); self._pos = s['pos']

In [ ]:
# ── 5. nanoGPT model (verbatim from karpathy/nanoGPT/model.py) ────────────────
# DO NOT modify — must be identical across all optimizer ablation runs.

import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias   = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_attn      = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj      = nn.Linear(config.n_embd, config.n_embd,     bias=config.bias)
        self.attn_dropout  = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head; self.n_embd = config.n_embd; self.dropout = config.dropout
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer('bias', torch.tril(torch.ones(config.block_size, config.block_size))
                                         .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = self.attn_dropout(F.softmax(att, dim=-1))
            y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp  = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int   = 1024
    vocab_size: int   = 50304   # GPT-2 50257 padded to next mult of 64
    n_layer:    int   = 8       # 12→8 to stay ≤100M with weight tying
    n_head:     int   = 12
    n_embd:     int   = 768
    dropout:    float = 0.0
    bias:       bool  = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight   # weight tying
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
        n = self.get_num_params()
        assert n <= 100_000_000, f'Over 100M: {n:,}'
        print(f'Parameters: {n/1e6:.2f}M')

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h: x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            return logits, F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return self.lm_head(x[:, [-1], :]), None

In [ ]:
# ── 6. Optimizer implementations ─────────────────────────────────────────────

# ── Muon (single-GPU) ────────────────────────────────────────────────────────
# Applied to 2D+ weight matrices in transformer blocks.
# Embeddings, positional embeds, LayerNorm, biases → AdamW.

@torch.compile
def _zeropower_ns5(G: torch.Tensor) -> torch.Tensor:
    """Newton-Schulz orthogonalization (5 iterations, quintic polynomial)."""
    X = G.bfloat16()
    if G.size(-2) > G.size(-1): X = X.mT
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    a, b, c = 3.4445, -4.7750, 2.0315
    for _ in range(5):
        A = X @ X.mT
        X = a * X + (b * A + c * A @ A) @ X
    if G.size(-2) > G.size(-1): X = X.mT
    return X.to(G.dtype)

class Muon(torch.optim.Optimizer):
    """
    Single-GPU Muon: MomentUm Orthogonalized by Newton-Schulz.
    https://kellerjordan.github.io/posts/muon/
    Only for 2D+ weight matrices; use AdamW for all others.
    """
    def __init__(self, params, lr=0.025, weight_decay=0.0125, momentum=0.95):
        super().__init__(params, dict(lr=lr, weight_decay=weight_decay, momentum=momentum))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr, wd, mu = group['lr'], group['weight_decay'], group['momentum']
            for p in group['params']:
                if p.grad is None: continue
                g = p.grad
                state = self.state[p]
                if 'buf' not in state: state['buf'] = torch.zeros_like(g)
                buf = state['buf']
                buf.lerp_(g, 1 - mu)            # EMA of gradients
                g = g.lerp(buf, mu)             # Nesterov momentum
                g = _zeropower_ns5(g)           # orthogonalize
                g = g * max(1.0, p.size(-2) / p.size(-1)) ** 0.5  # shape-based LR scaling
                p.mul_(1 - lr * wd)             # weight decay
                p.add_(g, alpha=-lr)


# ── LR schedule (cosine with warmup, same shape for all optimizers) ───────────

def make_cosine_schedule(peak_lr, min_lr, warmup, total):
    """Returns get_lr(step) → float, given peak/min LR and warmup/total steps."""
    def get_lr(step):
        if step < warmup:
            return peak_lr * step / warmup
        if step >= total:
            return min_lr
        decay = (step - warmup) / (total - warmup)
        return min_lr + 0.5 * (1.0 + math.cos(math.pi * decay)) * (peak_lr - min_lr)
    return get_lr


def safe_exp(x):
    try: return math.exp(x)
    except OverflowError: return float('inf')


print('Optimizer implementations ready.')

In [ ]:
# ── 7. Build model + optimizer ────────────────────────────────────────────────
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = GPTConfig()
model  = GPT(config).to(device)

try:
    compiled = torch.compile(model, dynamic=False)
    print('torch.compile: enabled')
except Exception as e:
    compiled = model
    print(f'torch.compile: disabled ({e})')

# ── Build optimizer based on OPTIMIZER flag ───────────────────────────────────
all_params    = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params  = [p for _, p in all_params.items() if p.dim() >= 2]
nodecay_params = [p for _, p in all_params.items() if p.dim() < 2]

block_2d_ids = {id(p) for p in model.transformer.h.parameters() if p.dim() >= 2}
block_2d     = [p for p in model.parameters() if id(p) in block_2d_ids]
non_block    = [p for p in model.parameters() if id(p) not in block_2d_ids]

use_fused_adam = 'fused' in inspect.signature(torch.optim.AdamW).parameters

if OPTIMIZER == "adamw":
    optimizers = [torch.optim.AdamW(
        [{'params': decay_params,   'weight_decay': ADAMW_WD},
         {'params': nodecay_params, 'weight_decay': 0.0}],
        lr=ADAMW_LR, betas=ADAMW_BETAS,
        **(dict(fused=True) if use_fused_adam else {})
    )]
    lr_schedules = [make_cosine_schedule(ADAMW_LR, ADAMW_MIN_LR, WARMUP_ITERS, NUM_STEPS)]
    print(f'AdamW: {sum(p.numel() for p in decay_params):,} decay + '
          f'{sum(p.numel() for p in nodecay_params):,} no-decay params')

elif OPTIMIZER == "muon_adam":
    non_block_2d = [p for p in non_block if p.dim() >= 2]
    non_block_1d = [p for p in non_block if p.dim() <  2]
    opt_muon = Muon(block_2d, lr=MUON_LR, weight_decay=MUON_WD, momentum=MUON_MU)
    opt_adam = torch.optim.AdamW(
        [{'params': non_block_2d, 'weight_decay': ADAMW_WD},
         {'params': non_block_1d, 'weight_decay': 0.0}],
        lr=MUON_ADAM_LR, betas=ADAMW_BETAS,
        **(dict(fused=True) if use_fused_adam else {})
    )
    optimizers   = [opt_muon, opt_adam]
    lr_schedules = [
        make_cosine_schedule(MUON_LR,      MUON_MIN_LR,      WARMUP_ITERS, NUM_STEPS),
        make_cosine_schedule(MUON_ADAM_LR, MUON_ADAM_MIN_LR, WARMUP_ITERS, NUM_STEPS),
    ]
    print(f'Muon:  {sum(p.numel() for p in block_2d):,} params (2D block weights)')
    print(f'AdamW: {sum(p.numel() for p in non_block):,} params (embed + norms + biases)')

elif OPTIMIZER == "soap":
    try:
        from heavyball import SOAP
        optimizers = [SOAP(
            [{'params': decay_params,   'weight_decay': SOAP_WD},
             {'params': nodecay_params, 'weight_decay': 0.0}],
            lr=SOAP_LR,
            betas=SOAP_BETAS,
            precondition_frequency=SOAP_PRECOND_FREQ,
        )]
        print(f'SOAP (heavyball): {sum(p.numel() for p in decay_params):,} decay params')
    except ImportError:
        raise ImportError('Run: !pip install heavyball  then restart and re-run this cell.')
    lr_schedules = [make_cosine_schedule(SOAP_LR, SOAP_MIN_LR, WARMUP_ITERS, NUM_STEPS)]

else:
    raise ValueError(f'Unknown OPTIMIZER: {OPTIMIZER!r}. Choose adamw / muon_adam / soap.')

for opt in optimizers:
    for grp in opt.param_groups:
        grp['initial_lr'] = grp['lr']

train_loader = DataLoader(train_files, MICRO_BATCH, SEQ_LEN)
val_loader   = DataLoader(val_files,   16, SEQ_LEN)

# Resume from checkpoint if available
start_step, train_losses, val_log = 0, [], []
if CKPT_PATH.exists():
    print(f'\nResuming from {CKPT_PATH}')
    # weights_only=False needed for SOAP: its state dict contains collections.defaultdict
    # which PyTorch 2.6 blocks under the default safe-load path. Safe here — our own file.
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    for i, opt in enumerate(optimizers):
        opt.load_state_dict(ckpt[f'opt_{i}'])
    train_loader.load_state_dict(ckpt['loader'])
    start_step   = ckpt['step']
    train_losses = ckpt.get('train_losses', [])
    val_log      = ckpt.get('val_log', [])
    print(f'Resumed at step {start_step}/{NUM_STEPS}')

if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME, resume='allow',
               config=dict(optimizer=OPTIMIZER, n_layer=config.n_layer,
                           n_embd=config.n_embd, micro_batch=MICRO_BATCH,
                           grad_accum=GRAD_ACCUM, num_steps=NUM_STEPS,
                           total_params=model.get_num_params()))
    print('W&B initialized.')

In [ ]:
# ── 8. Training loop ──────────────────────────────────────────────────────────
# Identical structure across all OPTIMIZER modes.
# The only difference is which optimizer(s) are stepped.

compiled.train()
t_start     = time.time()
t_step      = time.time()
tokens_seen = start_step * MICRO_BATCH * SEQ_LEN * GRAD_ACCUM

for step in range(start_step, NUM_STEPS + 1):

    # ── Validation ────────────────────────────────────────────────────────────
    if step % VAL_EVERY == 0 or step == NUM_STEPS:
        compiled.eval()
        vl = 0.0
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            for _ in range(20):
                _, loss = compiled(*next(val_loader))
                vl += loss.item()
        val_loss = vl / 20
        val_ppl  = safe_exp(val_loss)
        elapsed  = time.time() - t_start
        tok_s    = (step - start_step) * MICRO_BATCH * SEQ_LEN * GRAD_ACCUM / max(elapsed, 1)
        print(f'step {step:5d}/{NUM_STEPS}  val_loss {val_loss:.4f}  '
              f'val_ppl {val_ppl:.2f}  tok/s {tok_s:.0f}  {elapsed/60:.1f}min')
        val_log.append((step, val_loss, val_ppl))
        if USE_WANDB:
            wandb.log({'val_loss': val_loss, 'val_ppl': val_ppl, 'tokens': tokens_seen}, step=step)
        compiled.train()

    if step == NUM_STEPS:
        break

    # ── Checkpoint ────────────────────────────────────────────────────────────
    if step > 0 and step % CKPT_EVERY == 0:
        ckpt = {'step': step, 'model': model.state_dict(),
                'loader': train_loader.state_dict(),
                'train_losses': train_losses, 'val_log': val_log}
        for i, opt in enumerate(optimizers):
            ckpt[f'opt_{i}'] = opt.state_dict()
        torch.save(ckpt, CKPT_PATH)
        print(f'  ✓ checkpoint at step {step}')

    # ── Training step ─────────────────────────────────────────────────────────
    # Update LR for this step
    for opt, get_lr in zip(optimizers, lr_schedules):
        lr = get_lr(step)
        for grp in opt.param_groups:
            grp['lr'] = lr

    for opt in optimizers:
        opt.zero_grad(set_to_none=True)

    loss_sum = 0.0
    for _ in range(GRAD_ACCUM):
        X, Y = next(train_loader)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            _, loss = compiled(X, Y)
            (loss / GRAD_ACCUM).backward()
        loss_sum += loss.item() / GRAD_ACCUM

    # Gradient clipping — same for all optimizers
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

    for opt in optimizers:
        opt.step()

    tokens_seen += MICRO_BATCH * SEQ_LEN * GRAD_ACCUM
    train_losses.append(loss_sum)

    if step % 50 == 0 and step > 0:
        step_ms = (time.time() - t_step) / 50 * 1000
        lr_now  = lr_schedules[0](step)
        print(f'  step {step:5d}  train_loss {loss_sum:.4f}  lr {lr_now:.2e}  {step_ms:.0f}ms/step')
        t_step = time.time()
        if USE_WANDB:
            wandb.log({'train_loss': loss_sum, 'lr': lr_now, 'tokens': tokens_seen}, step=step)

print('\nTraining complete!')

In [ ]:
# ── 9. Full validation PPL ────────────────────────────────────────────────────
compiled.eval()
val_tokens = load_shard(val_files[0])
EVAL_BATCH = 32
total_loss, total_toks = 0.0, 0

with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
    for i in range(0, len(val_tokens) - EVAL_BATCH * SEQ_LEN, EVAL_BATCH * SEQ_LEN):
        chunk = val_tokens[i : i + EVAL_BATCH * SEQ_LEN + 1]
        if len(chunk) < EVAL_BATCH * SEQ_LEN + 1: break
        X = torch.from_numpy(chunk[:-1].astype(np.int64)).view(EVAL_BATCH, SEQ_LEN).cuda()
        Y = torch.from_numpy(chunk[1:].astype(np.int64)).view(EVAL_BATCH, SEQ_LEN).cuda()
        _, loss = compiled(X, Y)
        total_loss += loss.item() * EVAL_BATCH * SEQ_LEN
        total_toks += EVAL_BATCH * SEQ_LEN

avg_loss  = total_loss / total_toks
final_ppl = safe_exp(avg_loss)
print(f'Optimizer             : {OPTIMIZER}')
print(f'Val loss (full shard) : {avg_loss:.4f}')
print(f'Val PPL  (full shard) : {final_ppl:.2f}')
print(f'Total params          : {model.get_num_params():,} ({model.get_num_params()/1e6:.1f}M)')

In [ ]:
# ── 10. Compare multiple runs (run after collecting ≥2 val_logs) ──────────────
try:
    import matplotlib.pyplot as plt

    results = {}
    for opt_name in ['adamw', 'muon_adam', 'soap']:
        p = BASE_DIR / f'checkpoints_ablation_{opt_name}' / f'{opt_name}_latest.pt'
        if p.exists():
            # weights_only=False: SOAP state dicts contain collections.defaultdict
            ck = torch.load(p, map_location='cpu', weights_only=False)
            if ck.get('val_log'):
                results[opt_name] = ck['val_log']
                print(f'{opt_name}: {len(ck["val_log"])} val points, '
                      f'best PPL {min(r[2] for r in ck["val_log"]):.2f}')

    if results:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        colors = {'adamw': '#4C72B0', 'muon_adam': '#DD8452', 'soap': '#55A868'}
        for name, log in results.items():
            steps, losses, ppls = zip(*log)
            axes[0].plot(steps, losses, label=name, color=colors.get(name), linewidth=1.8)
            axes[1].plot(steps, ppls,   label=name, color=colors.get(name),
                         linewidth=1.8, marker='o', markersize=3)
        for ax, ylabel, title in zip(axes,
                                     ['Val Loss', 'Val PPL'],
                                     ['Validation Loss', 'Validation PPL']):
            ax.set(xlabel='Step', ylabel=ylabel, title=title)
            ax.legend(); ax.grid(alpha=0.3)
        plt.suptitle('Optimizer Ablation — nanoGPT arch, FineWebEDU, 5k steps', y=1.02)
        plt.tight_layout()
        out = CKPT_DIR.parent / 'optimizer_ablation_comparison.png'
        plt.savefig(out, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved {out}')
    else:
        print('No completed runs found yet. Run with each OPTIMIZER value first.')
except Exception as e:
    print(f'Plotting skipped: {e}')